In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
BASE_DIR = Path.cwd()
DATA_1 = BASE_DIR / "unemployment_india_cleaned.csv"
DATA_2 = BASE_DIR / "unemployment_rate_upto_nov2020_cleaned.csv"
area_data = pd.read_csv(DATA_1, parse_dates=["Date"])
state_data = pd.read_csv(DATA_2, parse_dates=["Date"])
rate = "Estimated Unemployment Rate (%)"
employment = "Estimated Employed"
participation = "Estimated Labour Participation Rate (%)"


In [ ]:
print("Area-level dataset shape:", area_data.shape)
print("State-level dataset shape:", state_data.shape)
print("\nArea-level columns:")
print(area_data.columns.tolist())
print("\nState-level columns:")
print(state_data.columns.tolist())


In [ ]:
print("Missing values in area-level dataset:")
print(area_data.isna().sum())
print("\nMissing values in state-level dataset:")
print(state_data.isna().sum())
print("\nDuplicate rows:")
print("Area-level:", area_data.duplicated().sum())
print("State-level:", state_data.duplicated().sum())


In [ ]:
print(area_data.describe(include="all").to_string())
print(state_data.describe(include="all").to_string())


In [ ]:
monthly_state = state_data.groupby("Date")[rate].mean().reset_index()
monthly_area = area_data.groupby(["Date", "Area"])[rate].mean().reset_index()
monthly_state


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(monthly_state["Date"], monthly_state[rate], marker="o")
plt.title("Average Unemployment Rate by Month (2020)")
plt.xlabel("Month")
plt.ylabel("Unemployment rate (%)")
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
for area in monthly_area["Area"].dropna().unique():
    subset = monthly_area[monthly_area["Area"] == area]
    plt.plot(subset["Date"], subset[rate], marker="o", label=area)
plt.title("Rural and Urban Unemployment Trends")
plt.xlabel("Month")
plt.ylabel("Unemployment rate (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
pre_covid = state_data.loc[state_data["Date"] < "2020-03-01", rate].mean()
covid_period = state_data.loc[state_data["Date"] >= "2020-03-01", rate].mean()
change = covid_period - pre_covid
print(f"Average rate before March 2020: {pre_covid:.2f}%")
print(f"Average rate from March to October 2020: {covid_period:.2f}%")
print(f"Difference: {change:.2f} percentage points")


In [ ]:
peak_month = monthly_state.loc[monthly_state[rate].idxmax()]
lowest_month = monthly_state.loc[monthly_state[rate].idxmin()]
print(f"Peak monthly average: {peak_month['Date'].strftime('%B %Y')} ({peak_month[rate]:.2f}%)")
print(f"Lowest monthly average: {lowest_month['Date'].strftime('%B %Y')} ({lowest_month[rate]:.2f}%)")


In [ ]:
state_average = state_data.groupby("Region")[rate].mean().sort_values(ascending=False)
print("Five highest state averages:")
print(state_average.head(5).round(2))
print("\nFive lowest state averages:")
print(state_average.tail(5).round(2))


In [ ]:
plt.figure(figsize=(10, 7))
state_average.sort_values().plot(kind="barh")
plt.title("Average Unemployment Rate by State/Region")
plt.xlabel("Average unemployment rate (%)")
plt.ylabel("State/region")
plt.tight_layout()
plt.show()


In [ ]:
monthly_area_pivot = monthly_area.pivot(index="Date", columns="Area", values=rate)
monthly_area_pivot


In [ ]:
area_data.to_csv(BASE_DIR / "unemployment_india_cleaned.csv", index=False)
state_data.to_csv(BASE_DIR / "unemployment_rate_upto_nov2020_cleaned.csv", index=False)
monthly_state.to_csv(BASE_DIR / "monthly_unemployment_summary.csv", index=False)
state_average.rename("Average Unemployment Rate (%)").to_csv(BASE_DIR / "state_unemployment_summary.csv")
